# Clipzi AI — Whisper model → Google Drive

Notebook ini **self-contained**: tidak perlu clone repo Clipzi yang private.

1. Pilih model di sel konfigurasi.
2. Klik **Runtime → Run all**.
3. Login dengan akun Google yang punya akses ke folder model.
4. Setelah selesai, kembali ke admin **Settings → Whisper Models → Scan Drive**.
5. Aktifkan model yang baru muncul.

Upload memakai kuota Google Drive akun Anda, bukan service account.


In [ ]:
# @title Konfigurasi model { display-mode: "form" }
MODEL_REPO = "openai/whisper-large-v3-turbo"  # @param ["openai/whisper-large-v3-turbo", "openai/whisper-large-v3", "Systran/faster-whisper-large-v3", "Systran/faster-whisper-small"]
SLUG = "large-v3-turbo"  # @param {type:"string"}
QUANTIZATION = "float16"  # @param ["float16", "int8", "int8_float16"]
OVERWRITE = False  # @param {type:"boolean"}

# Folder root registry model Clipzi (jangan diubah).
FOLDER_ID = "1H8eharRd98Poq8e2qYEkq9UYiOaPTNR4"


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "faster-whisper",
        "ctranslate2",
        "transformers",
        "huggingface_hub",
        "google-api-python-client",
        "google-auth",
    ]
)


In [ ]:
from google.colab import auth
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build

creds, _ = google.auth.default()
drive = build("drive", "v3", credentials=creds, cache_discovery=False)

folder = drive.files().get(
    fileId=FOLDER_ID,
    fields="id,name,mimeType,capabilities,webViewLink",
    supportsAllDrives=True,
).execute()

if folder.get("mimeType") != "application/vnd.google-apps.folder":
    raise RuntimeError("FOLDER_ID bukan folder Google Drive.")

caps = folder.get("capabilities") or {}
if not caps.get("canAddChildren"):
    raise RuntimeError("Akun ini tidak punya izin menulis ke folder model.")

print("Folder tujuan :", folder.get("name"))
print("Folder ID     :", folder.get("id"))
print("Link          :", folder.get("webViewLink"))


In [ ]:
import shutil
from pathlib import Path

from huggingface_hub import snapshot_download

WORK = Path("/content/clipzi_model")
SOURCE = WORK / "source"
CT2_DIR = WORK / "ct2"

if WORK.exists():
    shutil.rmtree(WORK)
SOURCE.mkdir(parents=True, exist_ok=True)

print("Mengunduh:", MODEL_REPO)
snapshot_download(repo_id=MODEL_REPO, local_dir=str(SOURCE))
print("Download selesai:", SOURCE)


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REQUIRED = ("model.bin", "config.json", "tokenizer.json", "preprocessor_config.json")
VOCAB = ("vocabulary.json", "vocabulary.txt")
PYTORCH = ("model.safetensors", "pytorch_model.bin")
TENSORFLOW = ("tf_model.h5", "model.ckpt.index")
COPY_EXTRA = ("tokenizer.json", "preprocessor_config.json", "vocabulary.json", "vocabulary.txt")


def names(path: Path) -> set[str]:
    return {item.name.lower() for item in path.iterdir() if item.is_file()}


def classify(path: Path) -> str:
    found = names(path)
    if any(marker in found for marker in TENSORFLOW):
        return "tensorflow"
    if "model.bin" in found:
        return "ct2"
    if any(marker in found for marker in PYTORCH):
        return "pytorch"
    return "unknown"


def validate(path: Path) -> list[str]:
    found = names(path)
    missing = [name for name in REQUIRED if name not in found]
    if not any(name in found for name in VOCAB):
        missing.append("vocabulary.json/vocabulary.txt")
    return missing


kind = classify(SOURCE)
print("Jenis model:", kind)
if kind == "tensorflow":
    raise RuntimeError("TensorFlow tidak didukung. Gunakan repo PyTorch atau CT2.")
if kind == "unknown":
    raise RuntimeError("Tidak ada model.bin/model.safetensors/pytorch_model.bin.")

if kind == "pytorch":
    if CT2_DIR.exists():
        shutil.rmtree(CT2_DIR)
    converter = shutil.which("ct2-transformers-converter")
    command = (
        [converter]
        if converter
        else [sys.executable, "-m", "ctranslate2.converters.transformers"]
    ) + [
        "--model", str(SOURCE),
        "--output_dir", str(CT2_DIR),
        "--quantization", QUANTIZATION,
        "--force",
    ]
    print("Konversi:", " ".join(command))
    try:
        result = subprocess.run(command, check=True, capture_output=True, text=True)
        print(result.stdout[-2000:])
    except subprocess.CalledProcessError as exc:
        print("CONVERTER_STDOUT:\n", (exc.stdout or "")[-3000:])
        print("CONVERTER_STDERR:\n", (exc.stderr or "")[-5000:])
        raise
    for name in COPY_EXTRA:
        candidate = SOURCE / name
        if candidate.exists() and not (CT2_DIR / name).exists():
            shutil.copy2(candidate, CT2_DIR / name)
else:
    CT2_DIR = SOURCE

missing = validate(CT2_DIR)
if missing:
    raise RuntimeError("Model CT2 belum lengkap: " + ", ".join(missing))

size = sum(item.stat().st_size for item in CT2_DIR.iterdir() if item.is_file())
print("CT2 siap:", CT2_DIR, f"({size / 1024**3:.2f} GiB)")


In [ ]:
import io
import json
import re
import time

from googleapiclient.http import MediaFileUpload, MediaIoBaseUpload

FOLDER_MIME = "application/vnd.google-apps.folder"


def list_children(parent_id: str):
    items = []
    token = None
    while True:
        kwargs = {
            "q": f"'{parent_id}' in parents and trashed=false",
            "pageSize": 1000,
            "fields": "nextPageToken,files(id,name,mimeType,size)",
            "supportsAllDrives": True,
            "includeItemsFromAllDrives": True,
        }
        if token:
            kwargs["pageToken"] = token
        response = drive.files().list(**kwargs).execute()
        items.extend(response.get("files", []))
        token = response.get("nextPageToken")
        if not token:
            return items


def find_child(parent_id: str, name: str):
    safe_name = name.replace("'", "\\'")
    result = drive.files().list(
        q=f"name='{safe_name}' and '{parent_id}' in parents and trashed=false",
        pageSize=10,
        fields="files(id,name,mimeType)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()
    return next(iter(result.get("files", [])), None)


def delete_tree(file_id: str):
    for child in list_children(file_id):
        if child.get("mimeType") == FOLDER_MIME:
            delete_tree(child["id"])
        else:
            drive.files().delete(fileId=child["id"], supportsAllDrives=True).execute()
    drive.files().delete(fileId=file_id, supportsAllDrives=True).execute()


def create_folder(parent_id: str, name: str):
    return drive.files().create(
        body={"name": name, "mimeType": FOLDER_MIME, "parents": [parent_id]},
        fields="id,name,webViewLink",
        supportsAllDrives=True,
    ).execute()


def upload_file(path, parent_id: str):
    media = MediaFileUpload(
        str(path),
        mimetype="application/octet-stream",
        resumable=True,
        chunksize=8 * 1024 * 1024,
    )
    request = drive.files().create(
        body={"name": path.name, "parents": [parent_id]},
        media_body=media,
        fields="id,name,size",
        supportsAllDrives=True,
    )
    response = None
    while request is not None:
        status, response = request.next_chunk()
        if status:
            print(f"  upload {path.name}: {int(status.progress() * 100)}%")
    return response


def upload_json(name: str, payload: dict, parent_id: str):
    data = json.dumps(payload, indent=2, ensure_ascii=False).encode("utf-8")
    media = MediaIoBaseUpload(io.BytesIO(data), mimetype="application/json", resumable=False)
    return drive.files().create(
        body={"name": name, "parents": [parent_id]},
        media_body=media,
        fields="id,name",
        supportsAllDrives=True,
    ).execute()


safe_slug = re.sub(r"[^A-Za-z0-9._-]+", "-", (SLUG or MODEL_REPO.split("/")[-1]).strip()).strip(".-")
if not safe_slug:
    raise RuntimeError("SLUG tidak valid.")

existing = find_child(FOLDER_ID, safe_slug)
if existing:
    if not OVERWRITE:
        raise RuntimeError(
            f"Folder '{safe_slug}' sudah ada. Set OVERWRITE=True kalau ingin mengganti."
        )
    print("OVERWRITE=True: menghapus folder lama", existing["id"])
    delete_tree(existing["id"])

target = create_folder(FOLDER_ID, safe_slug)
print("Folder model:", target["webViewLink"])

files = sorted(item for item in CT2_DIR.iterdir() if item.is_file())
for item in files:
    print("Upload:", item.name)
    upload_file(item, target["id"])

manifest = {
    "source_repo": MODEL_REPO,
    "source_kind": kind,
    "quantization": QUANTIZATION,
    "slug": safe_slug,
    "size_bytes": size,
    "files": [item.name for item in files],
    "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
upload_json("manifest.json", manifest, target["id"])

print("\nSELESAI")
print("Folder :", target["webViewLink"])
print("Langkah berikutnya: admin → Settings → Whisper Models → Scan Drive → Aktifkan.")
